In [1]:
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

from selenium.webdriver.support.ui import WebDriverWait

from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    WebDriverException
)

from datetime import datetime, timedelta

START_DATE = datetime(2026, 4, 1)
END_DATE = datetime(2026, 6, 30)
MAX_CONSECUTIVE_EMPTY = 10
START_INDEX = 1
OUTPUT_FILE = 'data_laporan_crm_2026_Q1.csv'

options = Options()
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--window-size=1920,1080')
options.page_load_strategy = 'eager'

driver = webdriver.Chrome(service=Service(), options=options)
driver.set_page_load_timeout(20)

### Mengambil Teks dari Element

In [2]:
def ambil_text(driver, element_id):
    try:
        text = driver.find_element(By.ID, element_id).text.strip()

        if text:
            return text

        return None

    except NoSuchElementException:
        return None

### Membersihkan Description

In [3]:
def bersihkan_deskripsi_dan_lokasi(deskripsi):
    if not deskripsi:
        return None, None

    teks = (deskripsi.replace('\r\n', '\n').replace('\r', '\n'))

    if 'Permasalahan:' in teks:
        teks = teks.split('Permasalahan:', 1)[1]

    if 'Kategori:' in teks:
        bagian = teks.split('Kategori:', 1)
        deskripsi_bersih = bagian[0].strip()
        sisa = bagian[1]

    else:
        deskripsi_bersih = teks.strip()
        sisa = ''

    lokasi = None
    if 'Lokasi:' in sisa:
        lokasi = sisa.split('Lokasi:', 1)[1].strip()

    deskripsi_bersih = ' '.join(deskripsi_bersih.split())

    if lokasi:
        lokasi = ' '.join(lokasi.split())

    if not deskripsi_bersih:
        deskripsi_bersih = None

    if not lokasi:
        lokasi = None

    return deskripsi_bersih, lokasi

### Mengambil Semua Data dari Halaman

In [4]:
def ambil_data_halaman(driver, nomor_laporan):
    deskripsi_raw = ambil_text(driver, 'description')
    kategori = ambil_text(driver, 'category')
    tanggal = ambil_text(driver, 'dateOfEntry')
    lokasi_website = ambil_text(driver, 'location')
    status_laporan = ambil_text(driver, 'status')

    (
        deskripsi_laporan,
        lokasi_dari_deskripsi
    ) = bersihkan_deskripsi_dan_lokasi(
        deskripsi_raw
    )

    if lokasi_dari_deskripsi:
        lokasi = lokasi_dari_deskripsi

    else:
        lokasi = lokasi_website

    ada_data = any([deskripsi_laporan, kategori, tanggal, lokasi, status_laporan])

    if not ada_data:
        return None

    data = {
        'nomor_laporan': nomor_laporan,
        'deskripsi_raw': deskripsi_raw,
        'deskripsi_laporan': deskripsi_laporan,
        'kategori': kategori,
        'tanggal': tanggal,
        'lokasi': lokasi,
        'status_laporan': status_laporan,
        'scraped_at': datetime.now().strftime(
            '%Y-%m-%d %H:%M:%S'
        )
    }

    return data

### Scrape Satu Laporan

In [5]:
def scrape_laporan(driver, nomor_laporan):
    url = (f'https://crm.jakarta.go.id/' f'laporan/{nomor_laporan}')
    print(f'\nMembuka: {nomor_laporan}')

    try:
        driver.get(url)

    except TimeoutException:
        print(f'! {nomor_laporan}: 'f'page load timeout')
        print('Mencoba membaca data yang ''sudah tersedia...')

        try:
            WebDriverWait(
                driver,
                3
            ).until(
                lambda d: (
                    ambil_text(
                        d,
                        'description'
                    ) is not None
                    or
                    ambil_text(
                        d,
                        'status'
                    ) is not None
                )
            )

        except TimeoutException:

            print(
                f'X {nomor_laporan}: '
                f'tidak ada data setelah timeout'
            )
            return None, 'error'


    except WebDriverException as e:

        print(f'! {nomor_laporan}: ' f'WebDriver error')
        print(f'  Detail: {str(e)[:150]}')
        return None, 'error'

    try:

        WebDriverWait(
            driver,
            10
        ).until(
            lambda d: (
                ambil_text(
                    d,
                    'description'
                ) is not None
                or
                ambil_text(
                    d,
                    'status'
                ) is not None
                or
                ambil_text(
                    d,
                    'category'
                ) is not None
            )
        )

    except TimeoutException:
        print(
            f'X {nomor_laporan}: '
            f'data tidak muncul'
        )
        return None, 'empty'

    try:
        data = ambil_data_halaman(driver, nomor_laporan)

    except WebDriverException as e:
        print(f'! {nomor_laporan}: ' f'error ketika mengambil data')
        print(f'  Detail: {str(e)[:150]}')
        return None, 'error'

    if data is None:
        print(f'X {nomor_laporan}: ' f'tidak ada data')
        return None, 'empty'

    print(f'V {nomor_laporan} berhasil')
    print(f"  Kategori : {data['kategori']}")
    print(f"  Tanggal  : {data['tanggal']}")
    print(f"  Lokasi   : {data['lokasi']}")
    print(f"  Status   : {data['status_laporan']}")

    return data, 'success'

### Proses Scrapping

In [6]:
results = []
current_date = START_DATE

try:
    while current_date <= END_DATE:
        day_prefix = current_date.strftime('%y%m%d')
        print('\n')
        print('=' * 70)
        print(f'SCRAPING TANGGAL: ' f'{current_date.strftime("%d-%m-%Y")}')
        print('=' * 70)

        index_per_day = START_INDEX
        consecutive_empty = 0

        while True:
            nomor_laporan = (f'JK' f'{day_prefix}' f'{str(index_per_day).zfill(4)}')
            laporan, status_scraping = scrape_laporan(driver, nomor_laporan)
            index_per_day += 1

            if status_scraping == 'success':
                results.append(laporan)
                consecutive_empty = 0
                print(f'-> Total berhasil: ' f'{len(results)}')

            elif status_scraping == 'empty':
                consecutive_empty += 1
                print(f'X ID tidak ditemukan ' f'atau kosong ' f'({consecutive_empty}/' f'{MAX_CONSECUTIVE_EMPTY})')

                if (consecutive_empty >= MAX_CONSECUTIVE_EMPTY):
                    print('\n' f'-> Ditemukan ' f'{MAX_CONSECUTIVE_EMPTY} ' f'ID kosong berturut-turut.')
                    print('-> Mengakhiri scraping ''untuk tanggal ini.')
                    break

            elif status_scraping == 'error':
                print('! Error teknis.')
                print('-> Error tidak dihitung ''sebagai empty.')
                continue

        print('\n' f'V Selesai untuk tanggal ' f'{current_date.strftime("%d-%m-%Y")}')
        print(f'  Jumlah total sementara: ' f'{len(results)}')
        current_date += timedelta(days=1)

finally:
    driver.quit()
    print('\nV Browser ditutup.')



SCRAPING TANGGAL: 01-04-2026

Membuka: JK2604010001
X JK2604010001: data tidak muncul
X ID tidak ditemukan atau kosong (1/10)

Membuka: JK2604010002
X JK2604010002: data tidak muncul
X ID tidak ditemukan atau kosong (2/10)

Membuka: JK2604010003
X JK2604010003: data tidak muncul
X ID tidak ditemukan atau kosong (3/10)

Membuka: JK2604010004
X JK2604010004: data tidak muncul
X ID tidak ditemukan atau kosong (4/10)

Membuka: JK2604010005
X JK2604010005: data tidak muncul
X ID tidak ditemukan atau kosong (5/10)

Membuka: JK2604010006
X JK2604010006: data tidak muncul
X ID tidak ditemukan atau kosong (6/10)

Membuka: JK2604010007
X JK2604010007: data tidak muncul
X ID tidak ditemukan atau kosong (7/10)

Membuka: JK2604010008
X JK2604010008: data tidak muncul
X ID tidak ditemukan atau kosong (8/10)

Membuka: JK2604010009
X JK2604010009: data tidak muncul
X ID tidak ditemukan atau kosong (9/10)

Membuka: JK2604010010
X JK2604010010: data tidak muncul
X ID tidak ditemukan atau kosong (10/10

In [7]:
df = pd.DataFrame(results)

if not df.empty:
    df = df[
        [
            'nomor_laporan',
            'deskripsi_raw',
            'deskripsi_laporan',
            'kategori',
            'tanggal',
            'lokasi',
            'status_laporan',
            'scraped_at'
        ]
    ]

print('\n')
print('=' * 70)
print('HASIL SCRAPING')
print('=' * 70)
print(f'Jumlah laporan berhasil: {len(df)}')

if not df.empty:
    print('\nPreview data:')
    print(df.head())

if not df.empty:
    jumlah_duplikat = df[
        'nomor_laporan'
    ].duplicated().sum()

    print('\nJumlah nomor laporan duplikat:', jumlah_duplikat)

if not df.empty:
    print('\nJumlah missing value per kolom:')

    print(df.isnull().sum())

if (not df.empty and 'status_laporan' in df.columns):
    print('\nDistribusi status laporan:')
    print(df['status_laporan'].value_counts(dropna=False))



HASIL SCRAPING
Jumlah laporan berhasil: 1372

Preview data:
  nomor_laporan                                      deskripsi_raw  \
0  JK2604020003  Permasalahan:\nKebijakan estimasi waktu kerja ...   
1  JK2604020011  Permasalahan:\nlampu jalan padam sudah hampir ...   
2  JK2604020013  Permasalahan:\nmusic kelabnya minta dikecilkan...   
3  JK2604020018  Permasalahan:\nTolong bantu di perbaiki lampun...   
4  JK2604020023  Permasalahan:\nIKLAN ROKOK\n\nIKLAN ROKOK IKLA...   

                                   deskripsi_laporan  \
0  Kebijakan estimasi waktu kerja permohonam peny...   
1  lampu jalan padam sudah hampir sebulan gelap d...   
2  music kelabnya minta dikecilkan ya kalau sudah...   
3  Tolong bantu di perbaiki lampunya mati di depa...   
4  IKLAN ROKOK IKLAN ROKOK IKLAN ROKOK IKLAN ROKO...   

                   kategori      tanggal  \
0            Bantuan Sosial  02 Apr 2026   
1          Jaringan Listrik  02 Apr 2026   
2   Gangguan dan Kebisingan  02 Apr 2026   
3   

### Simpan CSV

In [8]:
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print('\nV Data berhasil disimpan sebagai:')
print(OUTPUT_FILE)
print(f'\nTotal data akhir: {len(df)}')


V Data berhasil disimpan sebagai:
data_laporan_crm_2026_Q1.csv

Total data akhir: 1372
